# Libraries

In [54]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import sentencepiece as spm

In [55]:
dataset_dir = os.path.join('..','datasets')
dataset_path = os.path.join(dataset_dir, 'Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)

# Data Processing

In our exploratory data analysis (EDA), we've identified the columns that we will be working with, namely, Content, and Brand. However, before training the actual model, we'll need to format the data in numbers since that's how the model can understand and learn the patterns.

We'll create a copy of the original dataframe just in case we want a reference to the original dataset, then use the copy for any modifications and processing that we'll perform.

In [5]:
news_df = df[['Brand','Content','Label']].copy()
news_df

,Brand,Content,Label
0,Inquirer,Pollution caused by traditional cooking fuel i...,Credible
1,Manila Times,Justice Secretary Vitaliano Aguirre 2nd and Ph...,Credible
2,Inquirer,President Rodrigo Duterte on Monday night desc...,Credible
3,Manila Times,THE militant fisher folk group Pambansang Laka...,Credible
4,Inquirer,Magdalo Rep. Gary Alejano is willing to lead t...,Credible
...,...,...,...
22453,Get Real Philippines,"Indeed, everybody is shocked — just shocked! —...",Not Credible
22454,Manila Times,"A TOTAL of 132,259 individuals from 28,101 fam...",Credible
22455,Adobo Chronicles,Shortly after Rod Duterte announced there will...,Not Credible
22456,Adobo Chronicles,President Barack Obama met for the first time ...,Not Credible


There are 2 things we need to address in our dataset:
- Class Imbalance - The Credible label is twice as large as the Not Credible label which may introduce label bias to the model where they predict "Credible" for majority of the dataset.

- String to Number - The model can't understand strings, so, we'll have to convert the strings to numbers, where each word or symbol is a unique number.

## Addressing the Class Imbalance

In [6]:
news_df.Label.value_counts()

Label
Credible        14802
Not Credible     7656
Name: count, dtype: int64

Let's first begin by defining the majority class and the minority class.

In [33]:
majority = news_df[news_df['Label'] == 'Credible']
minority = news_df[news_df['Label'] == 'Not Credible']
print(f'The majority has a length of {len(majority)} while the minority has {len(minority)}')
print(f'The difference between the two is {len(majority) - len(minority)}.')
print(f'The minority is about {round(len(minority)/len(majority) * 100, 2)}% of the majority')

The majority has a length of 14802 while the minority has 7656
The difference between the two is 7146.
The minority is about 51.72% of the majority


To resolve this, we can upsample the minority class to match that of the majority class, then concatenate them.

In [44]:
news_df_upsampled = pd.concat([
    majority,
    minority.sample(len(majority), replace = True)
]).sample(frac = 1, random_state = 42).reset_index(drop = True) # Shuffle Dataset

news_df_upsampled.Label.value_counts()

Label
Not Credible    14802
Credible        14802
Name: count, dtype: int64

In [45]:
news_df_upsampled.head()

,Brand,Content,Label
0,Adobo Chronicles,Oprah Winfrey is the undisputed “Queen of All ...,Not Credible
1,Get Real Philippines,"The notion of the “Silent Majority”, which the...",Not Credible
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,Credible
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,Credible
4,Adobo Chronicles,If there is one thing Donald Trump isn’t willi...,Not Credible


## Addressing String Encoding (String-to-Number Conversion)

In this section, we mainly deal with 2 things:
1. The Label encoding into numerical values
2. Brand and Content encoding into numerical values

### Label Encoding

Encoding the label is quite easy as we only need to create a dictionary of all the unique classes in our label. Our label consist only of 2 classes: Credible and Not Credible.

Considering that we are making a fake news detection model, we would want to answer the question "Is this news fake?" which would mean that a True (1) would mean it is fake, and False (0) would indicate it is not fake. We can convert using these by creating a dictionary and applying it to the dataset.

In [46]:
label_to_idx = {
    'Credible': 0,
    'Not Credible': 1
}

news_df_upsampled['Label'] = news_df_upsampled['Label'].map(label_to_idx)
news_df_upsampled.head()

,Brand,Content,Label
0,Adobo Chronicles,Oprah Winfrey is the undisputed “Queen of All ...,1
1,Get Real Philippines,"The notion of the “Silent Majority”, which the...",1
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,0
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,0
4,Adobo Chronicles,If there is one thing Donald Trump isn’t willi...,1


### Brand and Content Encoding

Now, we have to encode the content and brand into numerical values.

For this encoding, we'll utilize the Byte-Pair Encoding (BPE) tokenizer so that we can conserve memory while retaining the most information from each character used in the strings.

However, before applying the BPE Tokenizer, we'll have to combine the Brand with the Content. We can combine this by applying the Brand as a form of header or starter to the sequence, this way, the model can recognize the first parts of the sequence as the "author". We can accomplish this by adding the Brand and Content together with a colon and space after the Brand, which would look like the following:

Brand: Content

In [47]:
news_df_upsampled['Brand and Content'] = news_df_upsampled['Brand'] + ': ' + news_df_upsampled['Content']
news_df_upsampled.head()

,Brand,Content,Label,Brand and Content
0,Adobo Chronicles,Oprah Winfrey is the undisputed “Queen of All ...,1,Adobo Chronicles: Oprah Winfrey is the undispu...
1,Get Real Philippines,"The notion of the “Silent Majority”, which the...",1,Get Real Philippines: The notion of the “Silen...
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,0,Inquirer: The Philippine Drug Enforcement Agen...
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,0,Manila Times: Low-cost carrier Cebu Pacific sa...
4,Adobo Chronicles,If there is one thing Donald Trump isn’t willi...,1,Adobo Chronicles: If there is one thing Donald...


In applying the BPE tokenizer, we need to make sure that we only train it on the training set. So, we need to first separate the dataset into train and tests.

We're only interested in using the 'Brand and Content' and 'Label' columns, so we'll use the train_test_split() function to separate only these two columns.

In [50]:
train, test = train_test_split(
    news_df_upsampled[['Brand and Content', 'Label']], 
    test_size = 0.2,
    random_state = 42
)

In [52]:
display(train.head())
display(test.head())

,Brand and Content,Label
13271,Adobo Chronicles: Imagine being an owner of a ...,1
7552,Inquirer: No sex tape purportedly featuring Se...,0
11813,"Inquirer: Echoing Malacañang’s reaction, the s...",0
20905,Duterte Daily Stories: Nagising ako mula sa na...,1
15973,Pinoytrending Altervista: Senate’s Majority Fl...,1


,Brand and Content,Label
27664,"GRPundit: PNP, Friend or Foe?Hoodlums in unifo...",1
28750,Adobo Chronicles: Despite being considered the...,1
5115,Adobo Chronicles: Congratulations to the newly...,1
12401,Inquirer: The investigation by the Department ...,0
17123,Pilipinas Online Updates: National Bureau of I...,1


Now that we have the split, we have to export it to a text file called corpus so that we can plug it into the sentencepiece trainer. We'll write the brand and content of our trainset in a corpus.txt file and store it under our datasets directory.

In [60]:
corpus_path = os.path.join(dataset_dir, 'corpus.txt')

with open(corpus_path, 'w', encoding = 'utf-8') as f:
    for row in train['Brand and Content']:
        f.write(row + '\n')

After creating our corpus file, we'll now train a BPE tokenizer using our own corpus.

In [65]:
vocab_size = 8000
bpe_model_path = os.path.join('..','models','bpe')

spm.SentencePieceTrainer.train(
    input = corpus_path,
    model_prefix = os.path.join(bpe_model_path, 'spm'),
    vocab_size = vocab_size,
    model_type = 'bpe',
    pad_id = 3,
)

In [66]:
tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(os.path.join(bpe_model_path, 'spm.model'))

True